In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import os
from pathlib import Path
from matplotlib.path import Path

## Data download ##

In [10]:
## Water discharge Glomma at Faukstad
area_glomma = 1500e6  # m2
df_discharge = pd.read_csv("..\\Data\\2.595.0-Vannføring-inst-v1.csv", skiprows=1, delimiter=';', decimal=',')
df_discharge.drop(columns=["Korrigert", "Kontrollert"], inplace=True)
df_discharge.columns = ['date', "Vannføring [m3/s]"]
df_discharge["date"] = pd.to_datetime(df_discharge["date"])
df_discharge.set_index(("date"), inplace=True)
df_discharge.columns = ["discharge [m3/s]"]
df_discharge.index = df_discharge.index.date
df_discharge.index = pd.to_datetime(df_discharge.index)
df_discharge.dropna(inplace=True)
df_discharge = df_discharge.groupby(df_discharge.index).mean()
df_discharge = df_discharge * 1000 * 3600 * 24 / area_glomma # m3/s to m/day
print("discharge is_unique:", df_discharge.index.is_unique)


## Waterlevel Faukstad data
df_waterlevel = pd.read_csv("..\\Data\\2.595.0-Vannstand-inst-v1.csv", skiprows=1, delimiter=';', decimal=',')
df_waterlevel.drop(columns=["Korrigert", "Kontrollert"], inplace=True)
df_waterlevel["Tidspunkt"] = pd.to_datetime(df_waterlevel["Tidspunkt"])
df_waterlevel.set_index(("Tidspunkt"), inplace=True)
df_waterlevel.columns = ["waterlevel [cm]"]
df_waterlevel.index = df_waterlevel.index.date
df_waterlevel.index = pd.to_datetime(df_waterlevel.index)
df_waterlevel = df_waterlevel.groupby(df_waterlevel.index).mean()
df_waterlevel.columns = ["waterlevel [m]"]
df_waterlevel.dropna(inplace=True)


## Evaporation data
df_evap = pd.read_csv("..\Data\evap.csv", parse_dates=True, skiprows = 23, delimiter="\\s+") 
df_evap.columns = ["date", "evap [mm/day]", "Nan", "NaN"]
df_evap.drop(columns=["Nan", "NaN"], inplace=True)
df_evap["date"] = pd.to_datetime(df_evap["date"], format="%Y%m%d")
df_evap.set_index("date", inplace=True)
df_evap.index = pd.to_datetime(df_evap.index)
df_evap = df_evap.groupby(df_evap.index).mean()
df_evap.dropna(inplace=True)


## Precipitation data
df_precip = pd.read_csv("..\\Data\\prcp_obs.csv",
    delim_whitespace=True,   # kolommen gescheiden door spaties
    header=None,             # er is geen echte header
    comment="#",             # regels die met # beginnen negeren
    names=["date", "rainfall [mm/day]"]
)
df_precip.set_index("date", inplace=True)
df_precip.index = pd.to_datetime(df_precip.index, format="%Y%m%d")
df_precip = df_precip.groupby(df_precip.index).mean()
df_precip.dropna(inplace=True)


## Radiation data
df_rad = pd.read_csv("..\\Data\\rad_obs.csv",
    delim_whitespace=True,   # kolommen gescheiden door spaties
    header=None,             # er is geen echte header
    comment="#",             # regels die met # beginnen negeren
    names=["date", "qq [W/m2]"]
)
df_rad['date'] = pd.to_datetime(df_rad['date'], format="%Y%m%d")
df_rad.set_index("date", inplace=True)
df_rad.index = df_rad.index.date
df_rad.index = pd.to_datetime(df_rad.index)
df_rad = df_rad.groupby(df_rad.index).mean()
df_rad.dropna(inplace=True)


## Temperature data
df_temp = pd.read_csv("..\\Data\\tmean_obs.csv",
    delim_whitespace=True,   # kolommen gescheiden door spaties
    header=None,             # er is geen echte header
    comment="#",             # regels die met # beginnen negeren
    names=["date", "temp [C]"]
)
df_temp['date'] = pd.to_datetime(df_temp['date'], format="%Y%m%d")
df_temp.set_index("date", inplace=True)
df_temp.index = df_temp.index.date
df_temp.index = pd.to_datetime(df_temp.index)
df_temp = df_temp.groupby(df_temp.index).mean()
df_temp.dropna(inplace=True);




<>:31: SyntaxWarning: invalid escape sequence '\D'
<>:31: SyntaxWarning: invalid escape sequence '\D'
C:\Users\joppe\AppData\Local\Temp\ipykernel_23364\633961122.py:31: SyntaxWarning: invalid escape sequence '\D'
  df_evap = pd.read_csv("..\Data\evap.csv", parse_dates=True, skiprows = 23, delimiter="\\s+")


discharge is_unique: True


C:\Users\joppe\AppData\Local\Temp\ipykernel_23364\633961122.py:42: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_precip = pd.read_csv("..\\Data\\prcp_obs.csv",
C:\Users\joppe\AppData\Local\Temp\ipykernel_23364\633961122.py:55: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_rad = pd.read_csv("..\\Data\\rad_obs.csv",
C:\Users\joppe\AppData\Local\Temp\ipykernel_23364\633961122.py:70: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_temp = pd.read_csv("..\\Data\\tmean_obs.csv",


## Combining dataframes ##

In [11]:
# define which dataframes should overlap
dataframes_overlap = [df_discharge, df_temp, df_precip]
# Inner join: only take dates that are present in all dataframes
df_all_inner = pd.concat(dataframes_overlap, axis=1, join="inner")

## Save the file as CSV file ##

In [12]:
# Ensure the index is a DatetimeIndex
df = df_all_inner.copy()
df.index = pd.to_datetime(df.index)
print(df.columns)
df.columns = ['Q(mm/d)', 'TMean(C)', 'Precip(mm/day)']
# Year / month / day as separate columns
df["YYYY"] = df.index.year
df["MM"] = df.index.month
df["DD"] = df.index.day

# Put columns in the desired order
df = df[["YYYY", "MM", "DD", "Q(mm/d)", "TMean(C)", "Precip(mm/day)"]]

# Write out to CSV without index
output_path = os.path.join("..", "Data", "model_input.csv")
df.to_csv(output_path, index=False) ##



Index(['discharge [m3/s]', 'temp [C]', 'rainfall [mm/day]'], dtype='object')
